In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


## 1. Frame the Problem

**1. Define the objective: what are you trying to predict?**

The objective is to predict the sale price (in dollars) of residential homes in Ames, Iowa based on 79 explanatory variables describing various property characteristics.

**2. How should you frame this problem as a machine learning problem?**

This is a supervised learning regression problem. We have labeled training data (houses with known sale prices) and need to learn a function that maps input features (house characteristics) to a continuous output variable (sale price).

**3. How should performance be measured?**

Performance should be measured using Root Mean Squared Error (RMSE) on the logarithm of the predicted and actual sale prices, as this is the official evaluation metric for the Kaggle competition. This metric penalizes large errors more heavily and is less sensitive to outliers due to the log transformation. We should also use cross-validation on the training set to get a reliable estimate of model performance before testing.

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print("Training set shape:", train_df.shape)
print("Test set shape:", test_df.shape)
print()

print("Data types in training set:")
print(train_df.dtypes)
print()

print("Training data overview:")
print(train_df.info())
print()

print("First 5 rows of training data:")
print(train_df.head())

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import f_oneway

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("="*80)
print("SECTION 3: EXPLORATORY DATA ANALYSIS")
print("="*80)

# ============================================================================
# STEP 1: BASIC DATA UNDERSTANDING
# ============================================================================
print("\n" + "="*80)
print("STEP 1: BASIC DATA UNDERSTANDING")
print("="*80)

print(f"\nDataset shape: {train_df.shape}")
print(f"Number of features: {train_df.shape[1] - 1}") # Excluding Id
print(f"Number of samples: {train_df.shape[0]}")

# Separate feature types
numeric_features = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = train_df.select_dtypes(include=['object']).columns.tolist()

# Remove Id and SalePrice
numeric_features.remove('Id')
if 'SalePrice' in numeric_features:
    numeric_features.remove('SalePrice')

print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# ============================================================================
# STEP 2: TARGET VARIABLE ANALYSIS (SALEPRICE)
# ============================================================================
print("\n" + "="*80)
print("STEP 2: TARGET VARIABLE ANALYSIS (SALEPRICE)")
print("="*80)

print("\nBasic statistics:")
print(train_df['SalePrice'].describe())

# Test for normality
shapiro_stat, shapiro_p = stats.shapiro(train_df['SalePrice'].sample(min(5000, len(train_df))))
print(f"\nShapiro-Wilk test for normality:")
print(f"  Statistic: {shapiro_stat:.6f}")
print(f"  P-value: {shapiro_p:.6f}")
print(f"  Interpretation: {'Data is NOT normally distributed' if shapiro_p < 0.05 else 'Data is normally distributed'}")

print(f"\nSkewness: {train_df['SalePrice'].skew():.3f} (>0.5 indicates right skew)")
print(f"Kurtosis: {train_df['SalePrice'].kurtosis():.3f} (>3 indicates heavy tails)")

# Visualize target variable
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original distribution
axes[0].hist(train_df['SalePrice'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(train_df['SalePrice'].mean(), color='red', linestyle='--', label=f'Mean: ${train_df["SalePrice"].mean():,.0f}')
axes[0].axvline(train_df['SalePrice'].median(), color='green', linestyle='--', label=f'Median: ${train_df["SalePrice"].median():,.0f}')
axes[0].set_title(f'SalePrice Distribution\n(Skewness: {train_df["SalePrice"].skew():.2f})')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Log-transformed distribution
log_price = np.log(train_df['SalePrice'])
axes[1].hist(log_price, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].axvline(log_price.mean(), color='red', linestyle='--', label=f'Mean: {log_price.mean():.2f}')
axes[1].axvline(log_price.median(), color='green', linestyle='--', label=f'Median: {log_price.median():.2f}')
axes[1].set_title(f'Log(SalePrice) Distribution\n(Skewness: {log_price.skew():.2f})')
axes[1].set_xlabel('Log(Sale Price)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Q-Q plot
stats.probplot(train_df['SalePrice'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot: SalePrice vs Normal Distribution')

plt.tight_layout()
plt.show()

print("\n→ INSIGHT: SalePrice is right-skewed. Log transformation will be needed for modeling.")

# ============================================================================
# STEP 3: CORRELATION ANALYSIS - IDENTIFY IMPORTANT NUMERIC FEATURES
# ============================================================================
print("\n" + "="*80)
print("STEP 3: CORRELATION ANALYSIS - IDENTIFY IMPORTANT NUMERIC FEATURES")
print("="*80)

# Calculate correlations with SalePrice
correlations = train_df[numeric_features + ['SalePrice']].corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

print("\nTop 15 features positively correlated with SalePrice:")
print(correlations.head(15))

print("\nFeatures with negative correlation (bottom 10):")
print(correlations.tail(10))

# Visualize top correlations
fig, ax = plt.subplots(figsize=(14, 10))
top_20_corr = pd.concat([correlations.head(20)])
top_20_corr.plot(kind='barh', ax=ax, color=['green' if x > 0 else 'red' for x in top_20_corr])
ax.set_title('Top 20 Features Correlated with SalePrice')
ax.set_xlabel('Correlation Coefficient')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.axvline(x=0.5, color='blue', linestyle='--', linewidth=0.5, alpha=0.5, label='Strong correlation threshold')
ax.legend()
plt.tight_layout()
plt.show()

# Select important features for detailed analysis (correlation > 0.5)
important_numeric = correlations[correlations > 0.5].index.tolist()
print(f"\n→ IMPORTANT NUMERIC FEATURES (correlation > 0.5): {len(important_numeric)}")
print(important_numeric)

# Identify weak features (may drop these)
weak_features = correlations[abs(correlations) < 0.1].index.tolist()
print(f"\n→ WEAK NUMERIC FEATURES (|correlation| < 0.1): {len(weak_features)}")
print(f"   These may be dropped: {weak_features}")

# Correlation matrix of top features
top_features = correlations.head(10).index.tolist()
corr_matrix = train_df[top_features + ['SalePrice']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Top 10 Numeric Features + SalePrice')
plt.tight_layout()
plt.show()

# ============================================================================
# STEP 4: DEEP DIVE INTO IMPORTANT NUMERIC FEATURES
# ============================================================================
print("\n" + "="*80)
print("STEP 4: DETAILED ANALYSIS OF IMPORTANT NUMERIC FEATURES")
print("="*80)

# Analyze each important feature
for feature in important_numeric[:8]:  # Top 8 for detailed analysis
    print(f"\n--- {feature} ---")
    print(train_df[feature].describe())
    print(f"Skewness: {train_df[feature].skew():.3f}")
    print(f"Missing values: {train_df[feature].isnull().sum()} ({train_df[feature].isnull().sum()/len(train_df)*100:.2f}%)")

# Visualize distributions
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, feature in enumerate(important_numeric[:8]):
    axes[idx].hist(train_df[feature].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{feature}\n(Skew: {train_df[feature].skew():.2f}, Corr: {correlations[feature]:.2f})')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    
    # Add mean and median lines
    axes[idx].axvline(train_df[feature].mean(), color='red', linestyle='--', alpha=0.7, label='Mean')
    axes[idx].axvline(train_df[feature].median(), color='green', linestyle='--', alpha=0.7, label='Median')
    axes[idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 5: OUTLIER DETECTION FOR IMPORTANT FEATURES
# ============================================================================
print("\n" + "="*80)
print("STEP 5: OUTLIER DETECTION FOR IMPORTANT FEATURES")
print("="*80)

# Scatter plots: Important features vs SalePrice
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

outlier_summary = {}

for idx, feature in enumerate(important_numeric[:8]):
    # Scatter plot
    axes[idx].scatter(train_df[feature], train_df['SalePrice'], alpha=0.5, s=20)
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('SalePrice')
    axes[idx].set_title(f'SalePrice vs {feature}')
    
    # Identify outliers using IQR method
    Q1 = train_df[feature].quantile(0.25)
    Q3 = train_df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = train_df[(train_df[feature] < lower_bound) | (train_df[feature] > upper_bound)]
    outlier_summary[feature] = len(outliers)
    
    # Highlight outliers
    if len(outliers) > 0:
        axes[idx].scatter(outliers[feature], outliers['SalePrice'], 
                         color='red', s=50, alpha=0.7, label=f'{len(outliers)} outliers')
        axes[idx].legend()

plt.tight_layout()
plt.show()

print("\nOutlier counts (using IQR method):")
for feature, count in sorted(outlier_summary.items(), key=lambda x: x[1], reverse=True):
    pct = count / len(train_df) * 100
    print(f"  {feature}: {count} outliers ({pct:.2f}%)")

# Special attention to GrLivArea outliers (known data quality issues from competition)
print("\n--- CRITICAL OUTLIERS IN GrLivArea ---")
grliv_outliers = train_df[train_df['GrLivArea'] > 4000]
print(f"Houses with GrLivArea > 4000 sqft: {len(grliv_outliers)}")
if len(grliv_outliers) > 0:
    print(grliv_outliers[['GrLivArea', 'SalePrice', 'OverallQual', 'Neighborhood']].to_string())
    print("\n→ INSIGHT: Some very large houses have unusually low prices.")
    print("   Recommendation: Remove samples with GrLivArea > 4000 AND SalePrice < 300000")

# Identify specific problematic outliers
problematic_outliers = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)]
print(f"\n→ PROBLEMATIC OUTLIERS TO REMOVE: {len(problematic_outliers)} samples")
if len(problematic_outliers) > 0:
    print(f"   IDs: {problematic_outliers.index.tolist()}")

# ============================================================================
# STEP 6: IDENTIFY IMPORTANT CATEGORICAL FEATURES
# ============================================================================
print("\n" + "="*80)
print("STEP 6: IDENTIFY IMPORTANT CATEGORICAL FEATURES")
print("="*80)

# Use ANOVA to test relationship between categorical features and SalePrice
categorical_importance = {}

for cat_feature in categorical_features:
    # Get unique categories
    categories = train_df[cat_feature].dropna().unique()
    
    # Skip if too many missing values
    if train_df[cat_feature].isnull().sum() / len(train_df) > 0.5:
        continue
    
    # Group SalePrice by category
    groups = [train_df[train_df[cat_feature] == cat]['SalePrice'].dropna() for cat in categories]
    
    # Remove empty groups
    groups = [g for g in groups if len(g) > 0]
    
    if len(groups) > 1:
        # Perform ANOVA
        f_stat, p_value = f_oneway(*groups)
        categorical_importance[cat_feature] = {'f_stat': f_stat, 'p_value': p_value}

# Sort by F-statistic
categorical_importance_sorted = sorted(categorical_importance.items(), 
                                      key=lambda x: x[1]['f_stat'], 
                                      reverse=True)

print("\nTop 15 Categorical Features (by ANOVA F-statistic):")
print(f"{'Feature':<20} {'F-Statistic':<15} {'P-Value':<15} {'Significant?'}")
print("-" * 65)
for feature, stats_dict in categorical_importance_sorted[:15]:
    significant = "Yes ***" if stats_dict['p_value'] < 0.001 else "Yes **" if stats_dict['p_value'] < 0.01 else "Yes *" if stats_dict['p_value'] < 0.05 else "No"
    print(f"{feature:<20} {stats_dict['f_stat']:<15.2f} {stats_dict['p_value']:<15.6f} {significant}")

# Select top categorical features for detailed analysis
important_categorical = [feat for feat, _ in categorical_importance_sorted[:10]]
print(f"\n→ IMPORTANT CATEGORICAL FEATURES (Top 10): {important_categorical}")

# Identify weak categorical features
weak_categorical = [feat for feat, stats in categorical_importance_sorted if stats['p_value'] > 0.05]
print(f"\n→ WEAK CATEGORICAL FEATURES (p-value > 0.05): {len(weak_categorical)}")
if len(weak_categorical) > 0:
    print(f"   May consider dropping: {weak_categorical[:5]}")

# ============================================================================
# STEP 7: DEEP DIVE INTO IMPORTANT CATEGORICAL FEATURES
# ============================================================================
print("\n" + "="*80)
print("STEP 7: DETAILED ANALYSIS OF IMPORTANT CATEGORICAL FEATURES")
print("="*80)

# Classify as ordinal or nominal based on data description
ordinal_features = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['NA', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'FireplaceQu': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['NA', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PavedDrive': ['N', 'P', 'Y'],
    'PoolQC': ['NA', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence': ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],
    'LandContour': ['Low', 'HLS', 'Bnk', 'Lvl'],
    'LandSlope': ['Sev', 'Mod', 'Gtl']
}

print("\nClassification of important categorical features:")
for feature in important_categorical[:10]:
    if feature in ordinal_features:
        print(f"  {feature}: ORDINAL (order: {ordinal_features[feature]})")
    else:
        print(f"  {feature}: NOMINAL (no inherent order)")

# Analyze cardinality
print("\nCardinality of important categorical features:")
for feature in important_categorical[:10]:
    unique_count = train_df[feature].nunique()
    print(f"  {feature}: {unique_count} unique values")
    if unique_count > 20:
        print(f"    WARNING: High cardinality feature - may need special encoding")

# Visualize top categorical features
fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

for idx, feature in enumerate(important_categorical[:9]):
    if feature in train_df.columns:
        # Calculate median SalePrice for each category
        cat_stats = train_df.groupby(feature)['SalePrice'].agg(['median', 'count']).sort_values('median', ascending=False)
        
        # Box plot
        train_df.boxplot(column='SalePrice', by=feature, ax=axes[idx])
        axes[idx].set_title(f'SalePrice by {feature}')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('SalePrice ($)')
        axes[idx].tick_params(axis='x', rotation=45)
        
        # Print statistics
        print(f"\n{feature} - Impact on SalePrice:")
        print(cat_stats.head(10))

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.show()

# ============================================================================
# STEP 8: MISSING VALUES ANALYSIS WITH CONTEXT
# ============================================================================
print("\n" + "="*80)
print("STEP 8: MISSING VALUES ANALYSIS WITH CONTEXT")
print("="*80)

missing_data = train_df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

print(f"\nTotal features with missing values: {len(missing_data)}")
print("\nMissing values breakdown:\n")

# Categorize missing values
meaningful_na = []  # NA means "feature doesn't exist"
true_missing = []  # Actually missing data

for feature in missing_data.index:
    missing_count = missing_data[feature]
    missing_pct = (missing_count / len(train_df)) * 100
    
    # Classify based on feature name and missing percentage
    if feature in ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                   'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'GarageYrBlt',
                   'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']:
        category = "MEANINGFUL NA (no feature)"
        meaningful_na.append(feature)
    else:
        category = "TRUE MISSING (needs imputation)"
        true_missing.append(feature)
    
    print(f"{feature:<20} {missing_count:>5} ({missing_pct:>5.1f}%) → {category}")

print(f"\n→ INSIGHT:")
print(f"  - {len(meaningful_na)} features: NA means 'feature doesn't exist' (fill with 'None' or 0)")
print(f"  - {len(true_missing)} features: Truly missing data (need imputation strategy)")

# Features to potentially drop due to excessive missing values
excessive_missing = missing_data[missing_data / len(train_df) > 0.8]
if len(excessive_missing) > 0:
    print(f"\n→ FEATURES WITH >80% MISSING (consider dropping): {excessive_missing.index.tolist()}")

# Visualize missing values
missing_pct = (missing_data / len(train_df)) * 100
plt.figure(figsize=(14, 6))
missing_pct.plot(kind='bar', color=['green' if f in meaningful_na else 'red' for f in missing_pct.index])
plt.title('Missing Values: Green = Meaningful NA, Red = True Missing')
plt.ylabel('Percentage Missing (%)')
plt.xlabel('Features')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=50, color='orange', linestyle='--', label='50% threshold')
plt.axhline(y=80, color='darkred', linestyle='--', label='80% threshold (consider dropping)')
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================================
# STEP 9: MULTICOLLINEARITY ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("STEP 9: MULTICOLLINEARITY ANALYSIS")
print("="*80)

# Check for highly correlated feature pairs
print("\nHighly correlated numeric feature pairs (|correlation| > 0.8):")
corr_matrix = train_df[important_numeric].corr().abs()

# Get upper triangle
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find highly correlated pairs
high_corr_pairs = []
for column in upper_triangle.columns:
    for row in upper_triangle.index:
        corr_val = upper_triangle.loc[row, column]
        if corr_val > 0.8:
            # Determine which to keep
            corr_with_target = (correlations[column], correlations[row])
            recommendation = column if corr_with_target[0] > corr_with_target[1] else row
            high_corr_pairs.append((column, row, corr_val, recommendation))

if high_corr_pairs:
    print(f"\n{'Feature 1':<20} {'Feature 2':<20} {'Correlation':<15} {'Keep':<20} {'Reason'}")
    print("-" * 95)
    for col1, col2, corr_val, keep in high_corr_pairs:
        reason = f"Higher corr with target ({correlations[keep]:.3f})"
        print(f"{col1:<20} {col2:<20} {corr_val:<15.3f} {keep:<20} {reason}")
else:
    print("No highly correlated pairs found among important features.")

# Calculate VIF for multicollinearity
print("\n--- Variance Inflation Factor (VIF) Analysis ---")
print("Note: VIF > 10 indicates high multicollinearity\n")

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Prepare data for VIF (remove NaN values)
vif_data = train_df[important_numeric].dropna()

if len(vif_data) > 0:
    try:
        vif_df = pd.DataFrame()
        vif_df["Feature"] = vif_data.columns
        vif_df["VIF"] = [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_data.columns))]
        vif_df = vif_df.sort_values('VIF', ascending=False)
        
        print(vif_df.to_string(index=False))
        
        high_vif = vif_df[vif_df['VIF'] > 10]
        if len(high_vif) > 0:
            print(f"\n→ WARNING: {len(high_vif)} features have VIF > 10 (high multicollinearity)")
            print("  Consider removing these features or using regularization:")
            print(high_vif['Feature'].tolist())
    except Exception as e:
        print(f"VIF calculation error: {e}")
else:
    print("VIF calculation skipped (insufficient data)")

# ============================================================================
# STEP 10: FEATURE ENGINEERING OPPORTUNITIES
# ============================================================================
print("\n" + "="*80)
print("STEP 10: FEATURE ENGINEERING OPPORTUNITIES")
print("="*80)

print("\nProposed new features:")
print("\n1. TOTAL SQUARE FOOTAGE:")
print("   TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF")

print("\n2. AGE-RELATED FEATURES:")
print("   HouseAge = YrSold - YearBuilt")
print("   YearsSinceRemod = YrSold - YearRemodAdd")
print("   IsRemodeled = 1 if YearRemodAdd != YearBuilt else 0")

print("\n3. TOTAL BATHROOMS:")
print("   TotalBath = FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath")

print("\n4. TOTAL PORCH AREA:")
print("   TotalPorchSF = OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch")

print("\n5. BINARY 'HAS' FEATURES:")
print("   HasPool = 1 if PoolArea > 0 else 0")
print("   HasGarage = 1 if GarageArea > 0 else 0")
print("   HasBasement = 1 if TotalBsmtSF > 0 else 0")
print("   HasFireplace = 1 if Fireplaces > 0 else 0")
print("   Has2ndFloor = 1 if 2ndFlrSF > 0 else 0")

print("\n6. QUALITY INTERACTIONS:")
print("   OverallScore = OverallQual * OverallCond")
print("   QualityArea = OverallQual * GrLivArea")

print("\n7. POLYNOMIAL FEATURES (for key features):")
print("   GrLivArea_squared = GrLivArea^2")
print("   TotalBsmtSF_squared = TotalBsmtSF^2")

# Demonstrate some feature engineering
print("\n--- Example Feature Engineering (preview) ---")
demo_df = train_df.copy()

demo_df['TotalSF'] = demo_df['TotalBsmtSF'] + demo_df['1stFlrSF'] + demo_df['2ndFlrSF']
demo_df['HouseAge'] = demo_df['YrSold'] - demo_df['YearBuilt']
demo_df['TotalBath'] = (demo_df['FullBath'] + 0.5*demo_df['HalfBath'] + 
                         demo_df['BsmtFullBath'] + 0.5*demo_df['BsmtHalfBath'])

# Check correlation of new features with SalePrice
new_features = ['TotalSF', 'HouseAge', 'TotalBath']
print("\nCorrelation of engineered features with SalePrice:")
for feat in new_features:
    corr = demo_df[[feat, 'SalePrice']].corr().iloc[0, 1]
    print(f"  {feat}: {corr:.3f}")

# ============================================================================
# STEP 11: SUMMARY AND RECOMMENDATIONS FOR DATA PREPARATION
# ============================================================================
print("\n" + "="*80)
print("STEP 11: SUMMARY AND RECOMMENDATIONS")
print("="*80)

print("\n📊 KEY FINDINGS:")
print("-" * 80)

print("\n1. TARGET VARIABLE (SalePrice):")
print(f"  - Right-skewed (skewness: {train_df['SalePrice'].skew():.2f})")
print("  - Log transformation REQUIRED for modeling")
print("  - After log transform, distribution is approximately normal")

print("\n2. IMPORTANT NUMERIC FEATURES (for modeling):")
for i, feat in enumerate(important_numeric[:10], 1):
    print(f"  {i}. {feat} (correlation: {correlations[feat]:.3f})")

print("\n3. IMPORTANT CATEGORICAL FEATURES (for modeling):")
important_cat_with_type = []
for i, feat in enumerate(important_categorical[:10], 1):
    ordinal_flag = " [ORDINAL]" if feat in ordinal_features else " [NOMINAL]"
    print(f"  {i}. {feat}{ordinal_flag}")
    important_cat_with_type.append((feat, 'ordinal' if feat in ordinal_features else 'nominal'))

print("\n4. FEATURES TO DROP:")
print("  - Very weak correlation with target (|corr| < 0.1):")
if len(weak_features) > 0:
    print(f"    {weak_features}")
print("  - Excessive missing values (>80%):")
if len(excessive_missing) > 0:
    print(f"    {excessive_missing.index.tolist()}")
print("  - High multicollinearity (based on VIF analysis)")
if len(high_corr_pairs) > 0:
    redundant_features = list(set([pair[0] if pair[3] == pair[1] else pair[1] for pair in high_corr_pairs]))
    print(f"    Consider dropping: {redundant_features}")

print("\n5. OUTLIERS TO REMOVE:")
print("  - GrLivArea: Remove houses with >4000 sqft AND SalePrice < $300,000")
if len(problematic_outliers) > 0:
    print(f"    Affected IDs: {problematic_outliers.index.tolist()}")

print("\n6. MISSING VALUES STRATEGY:")
print("  - Meaningful NA (no feature exists): Fill with 'None' or 0")
print(f"    Features: {meaningful_na[:8]}")
print("  - True missing values: Impute using median (numeric) or mode (categorical)")
print(f"    Features: {true_missing}")

print("\n7. FEATURE ENGINEERING (TO IMPLEMENT IN SECTION 4):")
print("  ✓ Total square footage: TotalSF")
print("  ✓ House age: HouseAge, YearsSinceRemod")
print("  ✓ Total bathrooms: TotalBath")
print("  ✓ Has features: HasPool, HasGarage, HasBasement, etc.")
print("  ✓ Quality interactions: OverallScore, QualityArea")
print("  ✓ Total porch area: TotalPorchSF")

print("\n8. SKEWED FEATURES (need transformation):")
skewed_features = []
for feat in numeric_features:
    if feat in train_df.columns:
        skew_val = train_df[feat].skew()
        if abs(skew_val) > 0.75:
            skewed_features.append((feat, skew_val))

print("  Features with |skewness| > 0.75 (log transform recommended):")
for feat, skew in sorted(skewed_features, key=lambda x: abs(x[1]), reverse=True)[:15]:
    print(f"    {feat}: {skew:.2f}")

print("\n9. DATA PREPARATION CHECKLIST FOR SECTION 4:")
print("  ☐ Remove outliers (GrLivArea > 4000 AND SalePrice < 300000)")
print("  ☐ Handle missing values (meaningful NA vs true missing)")
print("  ☐ Create engineered features")
print("  ☐ Encode categorical features (OrdinalEncoder for ordinal, OneHotEncoder for nominal)")
print("  ☐ Log transform target variable (SalePrice)")
print("  ☐ Log/power transform skewed numeric features")
print("  ☐ Scale/standardize numeric features")
print("  ☐ Drop weak/redundant features")

# Create a feature selection summary for easy reference
print("\n10. RECOMMENDED FEATURE SET FOR MODELING:")
print("=" * 80)

recommended_numeric = [f for f in important_numeric if f not in weak_features]
recommended_categorical = [f for f in important_categorical[:15] if f not in excessive_missing.index.tolist()]

print(f"\nNUMERIC FEATURES ({len(recommended_numeric)}):")
print(recommended_numeric)

print(f"\nCATEGORICAL FEATURES ({len(recommended_categorical)}):")
for feat in recommended_categorical:
    feat_type = "ORDINAL" if feat in ordinal_features else "NOMINAL"
    print(f"  {feat} [{feat_type}]")

print(f"\nENGINEERED FEATURES (7):")
print("  TotalSF, HouseAge, YearsSinceRemod, TotalBath, TotalPorchSF, OverallScore, QualityArea")

print(f"\nTOTAL FEATURES FOR MODELING: ~{len(recommended_numeric) + len(recommended_categorical) + 7}")

print("\n" + "="*80)
print("EXPLORATORY DATA ANALYSIS COMPLETE")
print("="*80)
print("\nNext Step: Use these insights to build the data preparation pipeline in Section 4")

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

print("="*80)
print("SECTION 4: PREPARE THE DATA FOR ML")
print("="*80)

# ============================================================================
# STEP 1: CLEAN THE DATA - REMOVE OUTLIERS
# ============================================================================
print("\n" + "="*80)
print("STEP 1: CLEAN THE DATA - REMOVE OUTLIERS")
print("="*80)

print(f"\nOriginal training data shape: {train_df.shape}")

# Remove problematic outliers identified in EDA
# GrLivArea > 4000 AND SalePrice < 300000
outlier_indices = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)].index
print(f"Removing {len(outlier_indices)} outliers with IDs: {outlier_indices.tolist()}")

train_df_clean = train_df.drop(outlier_indices)
print(f"Cleaned training data shape: {train_df_clean.shape}")

# ============================================================================
# STEP 2: CUSTOM TRANSFORMERS FOR FEATURE ENGINEERING
# ============================================================================
print("\n" + "="*80)
print("STEP 2: CUSTOM TRANSFORMERS FOR FEATURE ENGINEERING")
print("="*80)

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Create new features from existing ones"""
    
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        print("\nCreating engineered features:")
        
        # 1. Total Square Footage
        X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
        print("  ✓ TotalSF = TotalBsmtSF + 1stFlrSF + 2ndFlrSF")
        
        # 2. Age-related features
        X['HouseAge'] = X['YrSold'] - X['YearBuilt']
        X['YearsSinceRemod'] = X['YrSold'] - X['YearRemodAdd']
        X['IsRemodeled'] = (X['YearRemodAdd'] != X['YearBuilt']).astype(int)
        print("  ✓ HouseAge, YearsSinceRemod, IsRemodeled")
        
        # 3. Total Bathrooms
        X['TotalBath'] = (X['FullBath'] + 0.5*X['HalfBath'] + 
                         X['BsmtFullBath'] + 0.5*X['BsmtHalfBath'])
        print("  ✓ TotalBath")
        
        # 4. Total Porch Area
        X['TotalPorchSF'] = (X['OpenPorchSF'] + X['EnclosedPorch'] + 
                            X['3SsnPorch'] + X['ScreenPorch'])
        print("  ✓ TotalPorchSF")
        
        # 5. Binary 'Has' features
        X['HasPool'] = (X['PoolArea'] > 0).astype(int)
        X['HasGarage'] = (X['GarageArea'] > 0).astype(int)
        X['HasBasement'] = (X['TotalBsmtSF'] > 0).astype(int)
        X['HasFireplace'] = (X['Fireplaces'] > 0).astype(int)
        X['Has2ndFloor'] = (X['2ndFlrSF'] > 0).astype(int)
        print("  ✓ HasPool, HasGarage, HasBasement, HasFireplace, Has2ndFloor")
        
        # 6. Quality Interactions
        X['OverallScore'] = X['OverallQual'] * X['OverallCond']
        X['QualityArea'] = X['OverallQual'] * X['GrLivArea']
        print("  ✓ OverallScore, QualityArea")
        
        return X


class MissingValueHandler(BaseEstimator, TransformerMixin):
    """Handle missing values based on their meaning"""
    
    def __init__(self):
        # Features where NA means "feature doesn't exist"
        self.meaningful_na_features = [
            'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
            'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond',
            'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'
        ]
        
        # Store imputers for true missing values
        self.numeric_imputer = None
        self.categorical_imputer = None
        
    def fit(self, X, y=None):
        # Fit imputers for true missing values
        numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
        categorical_cols = X.select_dtypes(include=['object']).columns
        
        # Exclude meaningful NA features from imputation
        numeric_to_impute = [col for col in numeric_cols if col not in self.meaningful_na_features]
        categorical_to_impute = [col for col in categorical_cols if col not in self.meaningful_na_features]
        
        # Fit imputers
        if numeric_to_impute:
            self.numeric_imputer = SimpleImputer(strategy='median')
            self.numeric_imputer.fit(X[numeric_to_impute])
        
        if categorical_to_impute:
            self.categorical_imputer = SimpleImputer(strategy='most_frequent')
            self.categorical_imputer.fit(X[categorical_to_impute])
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        print("\nHandling missing values:")
        
        # 1. Handle meaningful NA features
        for col in self.meaningful_na_features:
            if col in X.columns and X[col].isnull().any():
                if X[col].dtype == 'object':
                    X[col].fillna('None', inplace=True)
                else:
                    X[col].fillna(0, inplace=True)
        
        print(f"  ✓ Filled {len(self.meaningful_na_features)} meaningful NA features with 'None'/0")
        
        # 2. Impute true missing values
        numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
        categorical_cols = X.select_dtypes(include=['object']).columns
        
        numeric_to_impute = [col for col in numeric_cols if col not in self.meaningful_na_features]
        categorical_to_impute = [col for col in categorical_cols if col not in self.meaningful_na_features]
        
        if numeric_to_impute and self.numeric_imputer:
            X[numeric_to_impute] = self.numeric_imputer.transform(X[numeric_to_impute])
            print(f"  ✓ Imputed {len(numeric_to_impute)} numeric features with median")
        
        if categorical_to_impute and self.categorical_imputer:
            X[categorical_to_impute] = self.categorical_imputer.transform(X[categorical_to_impute])
            print(f"  ✓ Imputed {len(categorical_to_impute)} categorical features with mode")
        
        return X


class FeatureSelector(BaseEstimator, TransformerMixin):
    """Drop weak and redundant features identified in EDA"""
    
    def __init__(self):
        # Features to drop based on EDA
        self.weak_features = [
            'PoolArea', 'MoSold', '3SsnPorch', 'BsmtFinSF2', 'BsmtHalfBath',
            'MiscVal', 'LowQualFinSF', 'YrSold', 'OverallCond', 'MSSubClass'
        ]
        
        self.excessive_missing_features = [
            'PoolQC', 'MiscFeature', 'Alley', 'Fence'
        ]
        
        self.multicollinear_features = [
            'TotRmsAbvGrd',  # Correlated with GrLivArea
            '1stFlrSF',       # Correlated with TotalBsmtSF
            'GarageArea'      # Correlated with GarageCars
        ]
        
        # Also drop original features that we've engineered
        self.engineered_originals = [
            'YearBuilt', 'YearRemodAdd',  # Replaced by HouseAge, YearsSinceRemod
        ]
        
        self.features_to_drop = (self.weak_features + self.excessive_missing_features + 
                                self.multicollinear_features + self.engineered_originals)
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        print("\nDropping features:")
        
        existing_features_to_drop = [f for f in self.features_to_drop if f in X.columns]
        
        print(f"  - Weak features ({len([f for f in self.weak_features if f in X.columns])}): {[f for f in self.weak_features if f in X.columns][:5]}...")
        print(f"  - Excessive missing ({len([f for f in self.excessive_missing_features if f in X.columns])}): {[f for f in self.excessive_missing_features if f in X.columns]}")
        print(f"  - Multicollinear ({len([f for f in self.multicollinear_features if f in X.columns])}): {[f for f in self.multicollinear_features if f in X.columns]}")
        print(f"  - Engineered originals ({len([f for f in self.engineered_originals if f in X.columns])}): {[f for f in self.engineered_originals if f in X.columns]}")
        print(f"  ✓ Total features dropped: {len(existing_features_to_drop)}")
        
        X = X.drop(columns=existing_features_to_drop, errors='ignore')
        
        # Also drop Id if present
        if 'Id' in X.columns:
            X = X.drop(columns=['Id'])
        
        return X


class SkewnessTransformer(BaseEstimator, TransformerMixin):
    """Apply log transformation to highly skewed numeric features"""
    
    def __init__(self, skew_threshold=0.75):
        self.skew_threshold = skew_threshold
        self.skewed_features = []
        
    def fit(self, X, y=None):
        # Identify skewed numeric features
        numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
        
        for col in numeric_cols:
            if X[col].skew() > self.skew_threshold:
                self.skewed_features.append(col)
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        if self.skewed_features:
            print(f"\nApplying log transformation to {len(self.skewed_features)} skewed features:")
            print(f"  Features: {self.skewed_features[:10]}...")
            
            for col in self.skewed_features:
                if col in X.columns:
                    # Add 1 to avoid log(0)
                    X[col] = np.log1p(X[col])
        
        return X


# ============================================================================
# STEP 3: IDENTIFY FEATURE TYPES FOR ENCODING
# ============================================================================
print("\n" + "="*80)
print("STEP 3: IDENTIFY FEATURE TYPES FOR ENCODING")
print("="*80)

# Define ordinal features and their order (from data description)
ordinal_features_map = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PavedDrive': ['N', 'P', 'Y'],
    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],
    'LandSlope': ['Sev', 'Mod', 'Gtl']
}

# Important ordinal features to use (from EDA)
ordinal_features_to_use = [
    'ExterQual', 'KitchenQual', 'BsmtQual', 'GarageFinish', 
    'HeatingQC', 'BsmtExposure'
]

# Important nominal features to use (from EDA)
nominal_features_to_use = [
    'Foundation', 'CentralAir', 'Neighborhood', 'GarageType'
]

print(f"\nOrdinal features to encode ({len(ordinal_features_to_use)}):")
for feat in ordinal_features_to_use:
    print(f"  {feat}: {ordinal_features_map.get(feat, 'Unknown')}")

print(f"\nNominal features to encode ({len(nominal_features_to_use)}):")
for feat in nominal_features_to_use:
    print(f"  {feat}")

# ============================================================================
# STEP 4: BUILD THE PREPROCESSING PIPELINE
# ============================================================================
print("\n" + "="*80)
print("STEP 4: BUILD THE PREPROCESSING PIPELINE")
print("="*80)

# Step 1: Feature engineering (must happen first)
feature_engineering_pipeline = Pipeline([
    ('feature_engineer', FeatureEngineer()),
    ('missing_handler', MissingValueHandler()),
    ('feature_selector', FeatureSelector()),
    ('skewness_transformer', SkewnessTransformer(skew_threshold=0.75))
])

print("\nPreprocessing pipeline created:")
print("  1. Feature Engineering (create new features)")
print("  2. Missing Value Handling (meaningful NA vs true missing)")
print("  3. Feature Selection (drop weak/redundant features)")
print("  4. Skewness Transformation (log transform skewed features)")
print("  5. Encoding & Scaling (handled by ColumnTransformer)")

# ============================================================================
# STEP 5: PREPARE TRAINING DATA
# ============================================================================
print("\n" + "="*80)
print("STEP 5: PREPARE TRAINING DATA")
print("="*80)

# Separate features and target
X_train = train_df_clean.drop(columns=['SalePrice'])
y_train = train_df_clean['SalePrice']

print(f"\nOriginal X_train shape: {X_train.shape}")
print(f"Original y_train shape: {y_train.shape}")

# Apply log transformation to target variable
print("\nTransforming target variable:")
print(f"  Original SalePrice - Skewness: {y_train.skew():.3f}")
y_train_log = np.log(y_train)
print(f"  Log(SalePrice) - Skewness: {y_train_log.skew():.3f}")
print("  ✓ Target variable transformed")

# Apply feature engineering and preprocessing
print("\n" + "="*80)
print("APPLYING PREPROCESSING PIPELINE TO TRAINING DATA")
print("="*80)

X_train_processed = feature_engineering_pipeline.fit_transform(X_train)

print(f"\nProcessed X_train shape: {X_train_processed.shape}")
print(f"Features created/modified: {X_train_processed.shape[1] - X_train.shape[1] + len(feature_engineering_pipeline.named_steps['feature_selector'].features_to_drop)}")

# ============================================================================
# STEP 6: CREATE COLUMN TRANSFORMER FOR ENCODING AND SCALING
# ============================================================================
print("\n" + "="*80)
print("STEP 6: CREATE COLUMN TRANSFORMER FOR ENCODING AND SCALING")
print("="*80)

# Identify which features exist after preprocessing
available_ordinal = [f for f in ordinal_features_to_use if f in X_train_processed.columns]
available_nominal = [f for f in nominal_features_to_use if f in X_train_processed.columns]
numeric_features = X_train_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nAvailable features after preprocessing:")
print(f"  - Ordinal features: {len(available_ordinal)}")
print(f"  - Nominal features: {len(available_nominal)}")
print(f"  - Numeric features: {len(numeric_features)}")

# Create ordinal encoder with proper categories
ordinal_encoder = OrdinalEncoder(
    categories=[ordinal_features_map[feat] for feat in available_ordinal],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

# Create column transformer
column_transformer = ColumnTransformer(
    transformers=[
        ('ordinal', ordinal_encoder, available_ordinal),
        ('nominal', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), available_nominal),
        ('numeric', StandardScaler(), numeric_features)
    ],
    remainder='drop'  # Drop any remaining columns
)

# Fit and transform
X_train_encoded = column_transformer.fit_transform(X_train_processed)

print(f"\nFinal X_train shape after encoding and scaling: {X_train_encoded.shape}")

# Get feature names after transformation
ordinal_names = available_ordinal
nominal_names = column_transformer.named_transformers_['nominal'].get_feature_names_out(available_nominal)
all_feature_names = ordinal_names + list(nominal_names) + numeric_features

print(f"Total features in final dataset: {len(all_feature_names)}")

# ============================================================================
# STEP 7: PREPARE TEST DATA (using the same pipeline)
# ============================================================================
print("\n" + "="*80)
print("STEP 7: PREPARE TEST DATA")
print("="*80)

X_test = test_df.copy()
print(f"Original X_test shape: {X_test.shape}")

# Apply the same preprocessing
X_test_processed = feature_engineering_pipeline.transform(X_test)
X_test_encoded = column_transformer.transform(X_test_processed)

print(f"Final X_test shape after preprocessing: {X_test_encoded.shape}")

# ============================================================================
# STEP 8: SUMMARY OF DATA PREPARATION
# ============================================================================
print("\n" + "="*80)
print("STEP 8: SUMMARY OF DATA PREPARATION")
print("="*80)

print("\n✅ DATA PREPARATION COMPLETE!")
print("-" * 80)

print(f"\n📊 Training Data:")
print(f"  - Samples: {X_train_encoded.shape[0]}")
print(f"  - Features: {X_train_encoded.shape[1]}")
print(f"  - Target: Log-transformed SalePrice")

print(f"\n📊 Test Data:")
print(f"  - Samples: {X_test_encoded.shape[0]}")
print(f"  - Features: {X_test_encoded.shape[1]}")

print(f"\n🔧 Transformations Applied:")
print(f"  ✓ Removed {len(outlier_indices)} outliers")
print(f"  ✓ Created 16 engineered features")
print(f"  ✓ Handled missing values (meaningful NA + imputation)")
print(f"  ✓ Dropped {len(feature_engineering_pipeline.named_steps['feature_selector'].features_to_drop)} weak/redundant features")
print(f"  ✓ Log-transformed skewed features")
print(f"  ✓ Encoded {len(available_ordinal)} ordinal features")
print(f"  ✓ One-hot encoded {len(available_nominal)} nominal features")
print(f"  ✓ Standardized all numeric features")
print(f"  ✓ Log-transformed target variable")

print(f"\n📋 Feature Types in Final Dataset:")
print(f"  - Ordinal (encoded): {len(available_ordinal)}")
print(f"  - Nominal (one-hot): {len(nominal_names)} features from {len(available_nominal)} categories")
print(f"  - Numeric (scaled): {len(numeric_features)}")

print("\n" + "="*80)
print("READY FOR MODEL TRAINING (SECTION 5)")
print("="*80)

# Store for next sections
print("\nVariables ready for Section 5:")
print("  - X_train_encoded: Training features (preprocessed)")
print("  - y_train_log: Training target (log-transformed)")
print("  - X_test_encoded: Test features (preprocessed)")
print("  - column_transformer: Fitted transformer (for feature names)")
print("  - feature_engineering_pipeline: Fitted pipeline (for new data)")

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

print("="*80)
print("SECTION 5: EXPLORE DIFFERENT MODELS")
print("="*80)

# ============================================================================
# STEP 1: TRAIN UNREGULARIZED LINEAR MODEL
# ============================================================================
print("\n" + "="*80)
print("STEP 1: TRAIN UNREGULARIZED LINEAR MODEL")
print("="*80)

# Train Linear Regression (no regularization)
lr_model = LinearRegression()
lr_model.fit(X_train_encoded, y_train_log)

# Make predictions
y_train_pred_lr = lr_model.predict(X_train_encoded)

# Calculate metrics
train_rmse_lr = np.sqrt(mean_squared_error(y_train_log, y_train_pred_lr))
train_mae_lr = mean_absolute_error(y_train_log, y_train_pred_lr)
train_r2_lr = r2_score(y_train_log, y_train_pred_lr)

print("\n📊 Unregularized Linear Regression Performance:")
print(f"  Training RMSE: {train_rmse_lr:.4f}")
print(f"  Training MAE:  {train_mae_lr:.4f}")
print(f"  Training R²:   {train_r2_lr:.4f}")

# Cross-validation
cv_scores_lr = cross_val_score(lr_model, X_train_encoded, y_train_log, 
                                cv=5, scoring='neg_root_mean_squared_error')
cv_rmse_lr = -cv_scores_lr.mean()
cv_std_lr = cv_scores_lr.std()

print(f"\n📊 5-Fold Cross-Validation:")
print(f"  CV RMSE: {cv_rmse_lr:.4f} (+/- {cv_std_lr:.4f})")
print(f"  Individual folds: {-cv_scores_lr}")

# ============================================================================
# STEP 2: TRAIN REGULARIZED LINEAR MODEL (DEFAULT PARAMETERS)
# ============================================================================
print("\n" + "="*80)
print("STEP 2: TRAIN REGULARIZED LINEAR MODEL (RIDGE - DEFAULT PARAMETERS)")
print("="*80)

# Train Ridge Regression with default parameters
ridge_model = Ridge()  # default alpha=1.0
ridge_model.fit(X_train_encoded, y_train_log)

# Make predictions
y_train_pred_ridge = ridge_model.predict(X_train_encoded)

# Calculate metrics
train_rmse_ridge = np.sqrt(mean_squared_error(y_train_log, y_train_pred_ridge))
train_mae_ridge = mean_absolute_error(y_train_log, y_train_pred_ridge)
train_r2_ridge = r2_score(y_train_log, y_train_pred_ridge)

print("\n📊 Ridge Regression (alpha=1.0) Performance:")
print(f"  Training RMSE: {train_rmse_ridge:.4f}")
print(f"  Training MAE:  {train_mae_ridge:.4f}")
print(f"  Training R²:   {train_r2_ridge:.4f}")

# Cross-validation
cv_scores_ridge = cross_val_score(ridge_model, X_train_encoded, y_train_log, 
                                   cv=5, scoring='neg_root_mean_squared_error')
cv_rmse_ridge = -cv_scores_ridge.mean()
cv_std_ridge = cv_scores_ridge.std()

print(f"\n📊 5-Fold Cross-Validation:")
print(f"  CV RMSE: {cv_rmse_ridge:.4f} (+/- {cv_std_ridge:.4f})")
print(f"  Individual folds: {-cv_scores_ridge}")

# ============================================================================
# STEP 3: TRAIN DECISION TREE
# ============================================================================
print("\n" + "="*80)
print("STEP 3: TRAIN DECISION TREE (DEFAULT PARAMETERS)")
print("="*80)

# Train Decision Tree with default parameters
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train_encoded, y_train_log)

# Make predictions
y_train_pred_dt = dt_model.predict(X_train_encoded)

# Calculate metrics
train_rmse_dt = np.sqrt(mean_squared_error(y_train_log, y_train_pred_dt))
train_mae_dt = mean_absolute_error(y_train_log, y_train_pred_dt)
train_r2_dt = r2_score(y_train_log, y_train_pred_dt)

print("\n📊 Decision Tree Performance:")
print(f"  Training RMSE: {train_rmse_dt:.4f}")
print(f"  Training MAE:  {train_mae_dt:.4f}")
print(f"  Training R²:   {train_r2_dt:.4f}")
print(f"  Tree depth:    {dt_model.get_depth()}")
print(f"  Leaf nodes:    {dt_model.get_n_leaves()}")

# Cross-validation
cv_scores_dt = cross_val_score(dt_model, X_train_encoded, y_train_log, 
                                cv=5, scoring='neg_root_mean_squared_error')
cv_rmse_dt = -cv_scores_dt.mean()
cv_std_dt = cv_scores_dt.std()

print(f"\n📊 5-Fold Cross-Validation:")
print(f"  CV RMSE: {cv_rmse_dt:.4f} (+/- {cv_std_dt:.4f})")
print(f"  Individual folds: {-cv_scores_dt}")

# ============================================================================
# STEP 4: COMPARE MODEL PERFORMANCE
# ============================================================================
print("\n" + "="*80)
print("STEP 4: COMPARE MODEL PERFORMANCE")
print("="*80)

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Decision Tree'],
    'Train RMSE': [train_rmse_lr, train_rmse_ridge, train_rmse_dt],
    'Train MAE': [train_mae_lr, train_mae_ridge, train_mae_dt],
    'Train R²': [train_r2_lr, train_r2_ridge, train_r2_dt],
    'CV RMSE': [cv_rmse_lr, cv_rmse_ridge, cv_rmse_dt],
    'CV Std': [cv_std_lr, cv_std_ridge, cv_std_dt]
})

# Calculate overfitting metric (difference between train and CV)
comparison_df['Overfitting'] = comparison_df['Train RMSE'] - comparison_df['CV RMSE']

print("\n📊 Model Comparison Summary:")
print("="*80)
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training RMSE
models = comparison_df['Model']
axes[0].bar(models, comparison_df['Train RMSE'], color=['blue', 'green', 'red'], alpha=0.7)
axes[0].set_title('Training RMSE (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(comparison_df['Train RMSE']):
    axes[0].text(i, v + 0.002, f'{v:.4f}', ha='center', va='bottom', fontsize=10)

# Plot 2: Cross-Validation RMSE with std
axes[1].bar(models, comparison_df['CV RMSE'], yerr=comparison_df['CV Std'], 
            color=['blue', 'green', 'red'], alpha=0.7, capsize=5)
axes[1].set_title('Cross-Validation RMSE (Lower is Better)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(comparison_df['CV RMSE']):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', va='bottom', fontsize=10)

# Plot 3: Train R² Score
axes[2].bar(models, comparison_df['Train R²'], color=['blue', 'green', 'red'], alpha=0.7)
axes[2].set_title('Training R² Score (Higher is Better)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('R² Score')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylim([0, 1])
for i, v in enumerate(comparison_df['Train R²']):
    axes[2].text(i, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n🎯 KEY OBSERVATIONS:")
print("-" * 80)

best_train_idx = comparison_df['Train RMSE'].idxmin()
best_cv_idx = comparison_df['CV RMSE'].idxmin()
most_overfit_idx = comparison_df['Overfitting'].idxmax()

print(f"  ✓ Best Training Performance: {comparison_df.loc[best_train_idx, 'Model']}")
print(f"  ✓ Best Cross-Validation Performance: {comparison_df.loc[best_cv_idx, 'Model']}")
print(f"  ⚠ Most Overfitting: {comparison_df.loc[most_overfit_idx, 'Model']}")

if comparison_df.loc[2, 'Train R²'] > 0.99:
    print(f"\n  ⚠ Decision Tree shows signs of SEVERE OVERFITTING:")
    print(f"     - Perfect/near-perfect training score (R² = {comparison_df.loc[2, 'Train R²']:.4f})")
    print(f"     - Much worse cross-validation performance")
    print(f"     - Needs regularization (max_depth, min_samples_split, etc.)")

# ============================================================================
# STEP 5: ANALYZE MOST SIGNIFICANT VARIABLES
# ============================================================================
print("\n" + "="*80)
print("STEP 5: ANALYZE MOST SIGNIFICANT VARIABLES FOR EACH MODEL")
print("="*80)

# Get feature names
feature_names = all_feature_names

# --- LINEAR REGRESSION COEFFICIENTS ---
print("\n--- LINEAR REGRESSION: TOP 15 MOST IMPORTANT FEATURES ---")
print("(By absolute coefficient magnitude)")

lr_coefs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lr_model.coef_
})
lr_coefs['Abs_Coefficient'] = np.abs(lr_coefs['Coefficient'])
lr_coefs_sorted = lr_coefs.sort_values('Abs_Coefficient', ascending=False)

print("\n" + lr_coefs_sorted.head(15).to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Top positive coefficients
top_positive = lr_coefs_sorted.nlargest(10, 'Coefficient')
axes[0].barh(range(len(top_positive)), top_positive['Coefficient'], color='green', alpha=0.7)
axes[0].set_yticks(range(len(top_positive)))
axes[0].set_yticklabels(top_positive['Feature'])
axes[0].set_xlabel('Coefficient Value')
axes[0].set_title('Linear Regression: Top 10 Positive Coefficients', fontweight='bold')
axes[0].invert_yaxis()

# Top negative coefficients
top_negative = lr_coefs_sorted.nsmallest(10, 'Coefficient')
axes[1].barh(range(len(top_negative)), top_negative['Coefficient'], color='red', alpha=0.7)
axes[1].set_yticks(range(len(top_negative)))
axes[1].set_yticklabels(top_negative['Feature'])
axes[1].set_xlabel('Coefficient Value')
axes[1].set_title('Linear Regression: Top 10 Negative Coefficients', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# --- RIDGE REGRESSION COEFFICIENTS ---
print("\n--- RIDGE REGRESSION: TOP 15 MOST IMPORTANT FEATURES ---")
print("(By absolute coefficient magnitude)")

ridge_coefs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': ridge_model.coef_
})
ridge_coefs['Abs_Coefficient'] = np.abs(ridge_coefs['Coefficient'])
ridge_coefs_sorted = ridge_coefs.sort_values('Abs_Coefficient', ascending=False)

print("\n" + ridge_coefs_sorted.head(15).to_string(index=False))

# Compare with Linear Regression
print("\n--- COMPARISON: Linear vs Ridge Coefficients ---")
coef_comparison = pd.DataFrame({
    'Feature': feature_names,
    'Linear_Coef': lr_model.coef_,
    'Ridge_Coef': ridge_model.coef_
})
coef_comparison['Difference'] = np.abs(coef_comparison['Linear_Coef'] - coef_comparison['Ridge_Coef'])
coef_comparison_sorted = coef_comparison.sort_values('Difference', ascending=False)

print("\nTop 10 features with largest coefficient shrinkage (Ridge vs Linear):")
print(coef_comparison_sorted.head(10).to_string(index=False))

# --- DECISION TREE FEATURE IMPORTANCES ---
print("\n--- DECISION TREE: TOP 15 MOST IMPORTANT FEATURES ---")
print("(By feature importance)")

dt_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': dt_model.feature_importances_
})
dt_importances_sorted = dt_importances.sort_values('Importance', ascending=False)

print("\n" + dt_importances_sorted.head(15).to_string(index=False))

# Visualize
plt.figure(figsize=(12, 8))
top_20_dt = dt_importances_sorted.head(20)
plt.barh(range(len(top_20_dt)), top_20_dt['Importance'], color='purple', alpha=0.7)
plt.yticks(range(len(top_20_dt)), top_20_dt['Feature'])
plt.xlabel('Feature Importance')
plt.title('Decision Tree: Top 20 Most Important Features', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# --- COMPARE FEATURE IMPORTANCE ACROSS MODELS ---
print("\n--- FEATURE IMPORTANCE COMPARISON ACROSS MODELS ---")

importance_comparison = pd.DataFrame({
    'Feature': feature_names,
    'Linear_Importance': np.abs(lr_model.coef_),
    'Ridge_Importance': np.abs(ridge_model.coef_),
    'DT_Importance': dt_model.feature_importances_
})

# Normalize importances for comparison (scale to 0-1)
for col in ['Linear_Importance', 'Ridge_Importance', 'DT_Importance']:
    importance_comparison[col] = importance_comparison[col] / importance_comparison[col].max()

# Get top features from each model
linear_top = set(lr_coefs_sorted.head(10)['Feature'])
ridge_top = set(ridge_coefs_sorted.head(10)['Feature'])
dt_top = set(dt_importances_sorted.head(10)['Feature'])

# Find common important features
common_features = linear_top & ridge_top & dt_top
print(f"\nFeatures important in ALL three models ({len(common_features)}):")
print(f"  {common_features}")

# Features unique to each model
print(f"\nFeatures unique to Linear Regression top 10: {linear_top - ridge_top - dt_top}")
print(f"Features unique to Ridge Regression top 10: {ridge_top - linear_top - dt_top}")
print(f"Features unique to Decision Tree top 10: {dt_top - linear_top - ridge_top}")

# ============================================================================
# STEP 6: ANALYZE TYPES OF ERRORS
# ============================================================================
print("\n" + "="*80)
print("STEP 6: ANALYZE TYPES OF ERRORS THE MODELS MAKE")
print("="*80)

# Calculate residuals for each model
residuals_lr = y_train_log - y_train_pred_lr
residuals_ridge = y_train_log - y_train_pred_ridge
residuals_dt = y_train_log - y_train_pred_dt

print("\n📊 Residual Statistics:")
print("="*80)

residual_stats = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Decision Tree'],
    'Mean Residual': [residuals_lr.mean(), residuals_ridge.mean(), residuals_dt.mean()],
    'Std Residual': [residuals_lr.std(), residuals_ridge.std(), residuals_dt.std()],
    'Min Residual': [residuals_lr.min(), residuals_ridge.min(), residuals_dt.min()],
    'Max Residual': [residuals_lr.max(), residuals_ridge.max(), residuals_dt.max()],
})

print("\n" + residual_stats.to_string(index=False))

# --- RESIDUAL PLOTS ---
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

models_data = [
    ('Linear Regression', y_train_pred_lr, residuals_lr, 'blue'),
    ('Ridge Regression', y_train_pred_ridge, residuals_ridge, 'green'),
    ('Decision Tree', y_train_pred_dt, residuals_dt, 'red')
]

for idx, (name, predictions, residuals, color) in enumerate(models_data):
    # Row 1: Actual vs Predicted
    axes[0, idx].scatter(y_train_log, predictions, alpha=0.5, s=20, color=color)
    axes[0, idx].plot([y_train_log.min(), y_train_log.max()], 
                      [y_train_log.min(), y_train_log.max()], 
                      'r--', lw=2, label='Perfect Prediction')
    axes[0, idx].set_xlabel('Actual Log(SalePrice)')
    axes[0, idx].set_ylabel('Predicted Log(SalePrice)')
    axes[0, idx].set_title(f'{name}\nActual vs Predicted', fontweight='bold')
    axes[0, idx].legend()
    
    # Row 2: Residual Plot
    axes[1, idx].scatter(predictions, residuals, alpha=0.5, s=20, color=color)
    axes[1, idx].axhline(y=0, color='r', linestyle='--', lw=2)
    axes[1, idx].set_xlabel('Predicted Log(SalePrice)')
    axes[1, idx].set_ylabel('Residuals')
    axes[1, idx].set_title(f'{name}\nResidual Plot', fontweight='bold')

plt.tight_layout()
plt.show()

# --- RESIDUAL DISTRIBUTIONS ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, predictions, residuals, color) in enumerate(models_data):
    axes[idx].hist(residuals, bins=50, color=color, alpha=0.7, edgecolor='black')
    axes[idx].axvline(x=0, color='red', linestyle='--', lw=2, label='Zero Error')
    axes[idx].set_xlabel('Residuals')
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{name}\nResidual Distribution', fontweight='bold')
    axes[idx].legend()

plt.tight_layout()
plt.show()

# --- ERROR ANALYSIS ---
print("\n🔍 ERROR ANALYSIS:")
print("="*80)

# Identify large errors for each model
threshold = 0.3  # Large error threshold in log scale

for name, predictions, residuals, color in models_data:
    large_errors = np.abs(residuals) > threshold
    num_large_errors = large_errors.sum()
    pct_large_errors = (num_large_errors / len(residuals)) * 100
    
    print(f"\n--- {name} ---")
    print(f"  Samples with |error| > {threshold}: {num_large_errors} ({pct_large_errors:.2f}%)")
    
    if num_large_errors > 0:
        # Analyze large errors
        large_error_indices = np.where(large_errors)[0]
        
        # Underpredictions (actual > predicted)
        underpredictions = residuals > threshold
        num_under = underpredictions.sum()
        
        # Overpredictions (actual < predicted)
        overpredictions = residuals < -threshold
        num_over = overpredictions.sum()
        
        print(f"  Underpredictions (model too low): {num_under}")
        print(f"  Overpredictions (model too high): {num_over}")
        
        # Largest errors
        largest_error_idx = np.argmax(np.abs(residuals))
        print(f"  Largest absolute error: {np.abs(residuals.iloc[largest_error_idx]):.4f}")
        print(f"    Actual: {y_train_log.iloc[largest_error_idx]:.4f}, Predicted: {predictions[largest_error_idx]:.4f}")

# --- Q-Q PLOTS FOR RESIDUALS (Check normality assumption) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, predictions, residuals, color) in enumerate(models_data):
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[idx])
    axes[idx].set_title(f'{name}\nQ-Q Plot (Residuals)', fontweight='bold')
    axes[idx].get_lines()[0].set_color(color)
    axes[idx].get_lines()[0].set_markersize(4)

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 7: SUMMARY AND INSIGHTS
# ============================================================================
print("\n" + "="*80)
print("STEP 7: SUMMARY AND INSIGHTS")
print("="*80)

print("\n🎯 MODEL PERFORMANCE SUMMARY:")
print("-" * 80)

best_model_idx = comparison_df['CV RMSE'].idxmin()
best_model = comparison_df.loc[best_model_idx, 'Model']
best_cv_rmse = comparison_df.loc[best_model_idx, 'CV RMSE']

print(f"\n✅ BEST MODEL (Based on Cross-Validation RMSE): {best_model}")
print(f"   CV RMSE: {best_cv_rmse:.4f}")

print("\n📊 Model Characteristics:")
print("\n1. LINEAR REGRESSION (Unregularized):")
print(f"   - Simple, interpretable model")
print(f"   - No bias from regularization")
print(f"   - CV RMSE: {cv_rmse_lr:.4f}")
print(f"   - May have higher variance due to no regularization")

print("\n2. RIDGE REGRESSION (Regularized):")
print(f"   - Adds L2 penalty to reduce overfitting")
print(f"   - Shrinks coefficients toward zero")
print(f"   - CV RMSE: {cv_rmse_ridge:.4f}")
print(f"   - Better generalization than unregularized linear")

print("\n3. DECISION TREE:")
print(f"   - Non-linear, can capture complex patterns")
print(f"   - CV RMSE: {cv_rmse_dt:.4f}")
if train_r2_dt > 0.99:
    print(f"   - ⚠ SEVERELY OVERFITTING (Train R²: {train_r2_dt:.4f})")
    print(f"   - Needs constraint (max_depth, min_samples_split, etc.)")

print("\n🔑 KEY INSIGHTS:")
print("-" * 80)

print("\n1. FEATURE IMPORTANCE:")
if len(common_features) > 0:
    print(f"   ✓ Features consistently important across all models:")
    for feat in list(common_features)[:5]:
        print(f"     - {feat}")
else:
    print(f"   ⚠ No features are consistently in top 10 across all models")

print("\n2. ERROR PATTERNS:")
print(f"   - Linear models: Residuals approximately normally distributed")
print(f"   - Decision Tree: More complex error pattern, likely overfitting")

print("\n3. MODEL SELECTION:")
if cv_rmse_ridge < cv_rmse_lr and cv_rmse_ridge < cv_rmse_dt:
    print(f"   ✓ Ridge Regression performs best on cross-validation")
    print(f"   ✓ Regularization helps control overfitting")
elif cv_rmse_lr < cv_rmse_dt:
    print(f"   ✓ Linear Regression performs reasonably well")
    print(f"   ✓ Simple model may be sufficient for this problem")
else:
    print(f"   ⚠ Decision Tree has best CV performance but shows overfitting")
    print(f"   ✓ Hyperparameter tuning needed (Section 6)")

print("\n4. NEXT STEPS FOR SECTION 6:")
print(f"   ☐ Fine-tune Ridge/Lasso regularization parameter (alpha)")
print(f"   ☐ Fine-tune Decision Tree depth and complexity parameters")
print(f"   ☐ Try ensemble methods (Random Forest, Gradient Boosting)")

print("\n" + "="*80)
print("MODEL EXPLORATION COMPLETE - READY FOR SECTION 6 (FINE-TUNING)")
print("="*80)

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import uniform, randint

print("="*80)
print("SECTION 6: FINE-TUNE THE MODELS")
print("="*80)

# ============================================================================
# STEP 1: FINE-TUNE LINEAR MODEL USING RANDOMIZEDSEARCHCV
# ============================================================================
print("\n" + "="*80)
print("STEP 1: FINE-TUNE LINEAR MODELS USING RANDOMIZEDSEARCHCV")
print("="*80)

print("\nWe'll tune both Ridge and Lasso regression models to find the best regularized linear model.")

# --- RIDGE REGRESSION TUNING ---
print("\n" + "-"*80)
print("A. TUNING RIDGE REGRESSION")
print("-"*80)

# Define hyperparameter distribution for Ridge
ridge_param_dist = {
    'alpha': uniform(0.001, 100),  # Continuous uniform distribution from 0.001 to 100
}

print("\nHyperparameter search space for Ridge:")
print("  - alpha: Continuous uniform distribution [0.001, 100]")
print("  - Total iterations: 100")
print("  - Cross-validation folds: 5")

# Create Ridge model
ridge_model = Ridge(random_state=42)

# RandomizedSearchCV for Ridge
ridge_random_search = RandomizedSearchCV(
    estimator=ridge_model,
    param_distributions=ridge_param_dist,
    n_iter=100,  # Number of parameter settings sampled
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,  # Use all available cores
    verbose=1
)

print("\nFitting RandomizedSearchCV for Ridge (this may take a moment)...")
ridge_random_search.fit(X_train_encoded, y_train_log)

print(f"\n✅ RandomizedSearchCV for Ridge completed!")
print(f"\nBest parameters: {ridge_random_search.best_params_}")
print(f"Best CV RMSE: {-ridge_random_search.best_score_:.6f}")

# Get best Ridge model
best_ridge = ridge_random_search.best_estimator_

# Evaluate on training set
y_train_pred_ridge_tuned = best_ridge.predict(X_train_encoded)
train_rmse_ridge_tuned = np.sqrt(mean_squared_error(y_train_log, y_train_pred_ridge_tuned))
train_r2_ridge_tuned = r2_score(y_train_log, y_train_pred_ridge_tuned)

print(f"\nBest Ridge Model Performance:")
print(f"  Training RMSE: {train_rmse_ridge_tuned:.6f}")
print(f"  Training R²:   {train_r2_ridge_tuned:.6f}")
print(f"  CV RMSE:       {-ridge_random_search.best_score_:.6f}")

# --- LASSO REGRESSION TUNING ---
print("\n" + "-"*80)
print("B. TUNING LASSO REGRESSION")
print("-"*80)

# Define hyperparameter distribution for Lasso
lasso_param_dist = {
    'alpha': uniform(0.0001, 0.01),  # Lasso typically needs smaller alpha values
}

print("\nHyperparameter search space for Lasso:")
print("  - alpha: Continuous uniform distribution [0.0001, 0.01]")
print("  - Total iterations: 100")
print("  - Cross-validation folds: 5")

# Create Lasso model
lasso_model = Lasso(random_state=42, max_iter=10000)

# RandomizedSearchCV for Lasso
lasso_random_search = RandomizedSearchCV(
    estimator=lasso_model,
    param_distributions=lasso_param_dist,
    n_iter=100,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nFitting RandomizedSearchCV for Lasso (this may take a moment)...")
lasso_random_search.fit(X_train_encoded, y_train_log)

print(f"\n✅ RandomizedSearchCV for Lasso completed!")
print(f"\nBest parameters: {lasso_random_search.best_params_}")
print(f"Best CV RMSE: {-lasso_random_search.best_score_:.6f}")

# Get best Lasso model
best_lasso = lasso_random_search.best_estimator_

# Evaluate on training set
y_train_pred_lasso_tuned = best_lasso.predict(X_train_encoded)
train_rmse_lasso_tuned = np.sqrt(mean_squared_error(y_train_log, y_train_pred_lasso_tuned))
train_r2_lasso_tuned = r2_score(y_train_log, y_train_pred_lasso_tuned)

print(f"\nBest Lasso Model Performance:")
print(f"  Training RMSE: {train_rmse_lasso_tuned:.6f}")
print(f"  Training R²:   {train_r2_lasso_tuned:.6f}")
print(f"  CV RMSE:       {-lasso_random_search.best_score_:.6f}")

# Count non-zero coefficients in Lasso
non_zero_coefs = np.sum(best_lasso.coef_ != 0)
print(f"  Non-zero coefficients: {non_zero_coefs} / {len(best_lasso.coef_)}")
print(f"  Features eliminated: {len(best_lasso.coef_) - non_zero_coefs}")

# --- VISUALIZE RANDOMIZEDSEARCHCV RESULTS ---
print("\n" + "-"*80)
print("C. VISUALIZING RANDOMIZEDSEARCHCV RESULTS")
print("-"*80)

# Ridge results
ridge_results = pd.DataFrame(ridge_random_search.cv_results_)
ridge_results['rmse'] = -ridge_results['mean_test_score']

# Lasso results
lasso_results = pd.DataFrame(lasso_random_search.cv_results_)
lasso_results['rmse'] = -lasso_results['mean_test_score']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot Ridge alpha vs RMSE
axes[0].scatter(ridge_results['param_alpha'], ridge_results['rmse'], alpha=0.6, s=50)
axes[0].axhline(y=-ridge_random_search.best_score_, color='r', linestyle='--', 
                label=f'Best RMSE: {-ridge_random_search.best_score_:.6f}')
axes[0].axvline(x=ridge_random_search.best_params_['alpha'], color='g', linestyle='--',
                label=f'Best alpha: {ridge_random_search.best_params_["alpha"]:.4f}')
axes[0].set_xlabel('Alpha (Regularization Strength)', fontsize=12)
axes[0].set_ylabel('CV RMSE', fontsize=12)
axes[0].set_title('Ridge: Alpha vs Cross-Validation RMSE', fontsize=14, fontweight='bold')
axes[0].set_xscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Lasso alpha vs RMSE
axes[1].scatter(lasso_results['param_alpha'], lasso_results['rmse'], alpha=0.6, s=50, color='orange')
axes[1].axhline(y=-lasso_random_search.best_score_, color='r', linestyle='--',
                label=f'Best RMSE: {-lasso_random_search.best_score_:.6f}')
axes[1].axvline(x=lasso_random_search.best_params_['alpha'], color='g', linestyle='--',
                label=f'Best alpha: {lasso_random_search.best_params_["alpha"]:.6f}')
axes[1].set_xlabel('Alpha (Regularization Strength)', fontsize=12)
axes[1].set_ylabel('CV RMSE', fontsize=12)
axes[1].set_title('Lasso: Alpha vs Cross-Validation RMSE', fontsize=14, fontweight='bold')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 2: FINE-TUNE DECISION TREE USING GRIDSEARCHCV
# ============================================================================
print("\n" + "="*80)
print("STEP 2: FINE-TUNE DECISION TREE USING GRIDSEARCHCV")
print("="*80)

# Define hyperparameter grid for Decision Tree
dt_param_grid = {
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None]
}

print("\nHyperparameter search space for Decision Tree:")
print(f"  - max_depth: {dt_param_grid['max_depth']}")
print(f"  - min_samples_split: {dt_param_grid['min_samples_split']}")
print(f"  - min_samples_leaf: {dt_param_grid['min_samples_leaf']}")
print(f"  - max_features: {dt_param_grid['max_features']}")
print(f"\n  Total combinations: {7 * 4 * 4 * 3} = {7 * 4 * 4 * 3}")
print(f"  Cross-validation folds: 5")
print(f"  Total fits: {7 * 4 * 4 * 3 * 5} = {7 * 4 * 4 * 3 * 5}")

# Create Decision Tree model
dt_model = DecisionTreeRegressor(random_state=42)

# GridSearchCV for Decision Tree
dt_grid_search = GridSearchCV(
    estimator=dt_model,
    param_grid=dt_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("\nFitting GridSearchCV for Decision Tree (this will take a few moments)...")
dt_grid_search.fit(X_train_encoded, y_train_log)

print(f"\n✅ GridSearchCV for Decision Tree completed!")
print(f"\nBest parameters:")
for param, value in dt_grid_search.best_params_.items():
    print(f"  - {param}: {value}")
print(f"\nBest CV RMSE: {-dt_grid_search.best_score_:.6f}")

# Get best Decision Tree model
best_dt = dt_grid_search.best_estimator_

# Evaluate on training set
y_train_pred_dt_tuned = best_dt.predict(X_train_encoded)
train_rmse_dt_tuned = np.sqrt(mean_squared_error(y_train_log, y_train_pred_dt_tuned))
train_r2_dt_tuned = r2_score(y_train_log, y_train_pred_dt_tuned)

print(f"\nBest Decision Tree Model Performance:")
print(f"  Training RMSE: {train_rmse_dt_tuned:.6f}")
print(f"  Training R²:   {train_r2_dt_tuned:.6f}")
print(f"  CV RMSE:       {-dt_grid_search.best_score_:.6f}")
print(f"  Tree depth:    {best_dt.get_depth()}")
print(f"  Leaf nodes:    {best_dt.get_n_leaves()}")

# --- VISUALIZE GRIDSEARCHCV RESULTS ---
print("\n" + "-"*80)
print("VISUALIZING GRIDSEARCHCV RESULTS")
print("-"*80)

# Extract results
dt_results = pd.DataFrame(dt_grid_search.cv_results_)
dt_results['rmse'] = -dt_results['mean_test_score']

# Plot 1: Max Depth vs RMSE
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Max Depth
depth_results = dt_results.groupby('param_max_depth')['rmse'].agg(['mean', 'std'])
depth_results = depth_results.sort_index()
axes[0, 0].errorbar(range(len(depth_results)), depth_results['mean'], 
                     yerr=depth_results['std'], marker='o', capsize=5, linewidth=2)
axes[0, 0].set_xticks(range(len(depth_results)))
axes[0, 0].set_xticklabels([str(x) if x is not None else 'None' for x in depth_results.index])
axes[0, 0].set_xlabel('Max Depth', fontsize=12)
axes[0, 0].set_ylabel('CV RMSE', fontsize=12)
axes[0, 0].set_title('Decision Tree: Max Depth vs CV RMSE', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=-dt_grid_search.best_score_, color='r', linestyle='--', alpha=0.5)

# Min Samples Split
split_results = dt_results.groupby('param_min_samples_split')['rmse'].agg(['mean', 'std'])
split_results = split_results.sort_index()
axes[0, 1].errorbar(range(len(split_results)), split_results['mean'],
                     yerr=split_results['std'], marker='s', capsize=5, linewidth=2, color='orange')
axes[0, 1].set_xticks(range(len(split_results)))
axes[0, 1].set_xticklabels(split_results.index)
axes[0, 1].set_xlabel('Min Samples Split', fontsize=12)
axes[0, 1].set_ylabel('CV RMSE', fontsize=12)
axes[0, 1].set_title('Decision Tree: Min Samples Split vs CV RMSE', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=-dt_grid_search.best_score_, color='r', linestyle='--', alpha=0.5)

# Min Samples Leaf
leaf_results = dt_results.groupby('param_min_samples_leaf')['rmse'].agg(['mean', 'std'])
leaf_results = leaf_results.sort_index()
axes[1, 0].errorbar(range(len(leaf_results)), leaf_results['mean'],
                     yerr=leaf_results['std'], marker='^', capsize=5, linewidth=2, color='green')
axes[1, 0].set_xticks(range(len(leaf_results)))
axes[1, 0].set_xticklabels(leaf_results.index)
axes[1, 0].set_xlabel('Min Samples Leaf', fontsize=12)
axes[1, 0].set_ylabel('CV RMSE', fontsize=12)
axes[1, 0].set_title('Decision Tree: Min Samples Leaf vs CV RMSE', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(y=-dt_grid_search.best_score_, color='r', linestyle='--', alpha=0.5)

# Max Features
feature_results = dt_results.groupby('param_max_features')['rmse'].agg(['mean', 'std'])
feature_results = feature_results.sort_index()
axes[1, 1].errorbar(range(len(feature_results)), feature_results['mean'],
                     yerr=feature_results['std'], marker='D', capsize=5, linewidth=2, color='purple')
axes[1, 1].set_xticks(range(len(feature_results)))
axes[1, 1].set_xticklabels([str(x) if x is not None else 'None' for x in feature_results.index])
axes[1, 1].set_xlabel('Max Features', fontsize=12)
axes[1, 1].set_ylabel('CV RMSE', fontsize=12)
axes[1, 1].set_title('Decision Tree: Max Features vs CV RMSE', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=-dt_grid_search.best_score_, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Heatmap of top parameter combinations
print("\nTop 10 parameter combinations:")
top_10_results = dt_results.nsmallest(10, 'rmse')[['param_max_depth', 'param_min_samples_split', 
                                                     'param_min_samples_leaf', 'param_max_features', 'rmse']]
print(top_10_results.to_string(index=False))

# ============================================================================
# STEP 3: COMPARE TUNED MODELS WITH DEFAULT MODELS
# ============================================================================
print("\n" + "="*80)
print("STEP 3: COMPARE TUNED MODELS WITH DEFAULT MODELS")
print("="*80)

# Create comprehensive comparison
comparison_df = pd.DataFrame({
    'Model': [
        'Linear Regression (default)',
        'Ridge (default α=1.0)',
        'Ridge (tuned)',
        'Lasso (tuned)',
        'Decision Tree (default)',
        'Decision Tree (tuned)'
    ],
    'Train RMSE': [
        train_rmse_lr,
        train_rmse_ridge,
        train_rmse_ridge_tuned,
        train_rmse_lasso_tuned,
        train_rmse_dt,
        train_rmse_dt_tuned
    ],
    'Train R²': [
        train_r2_lr,
        train_r2_ridge,
        train_r2_ridge_tuned,
        train_r2_lasso_tuned,
        train_r2_dt,
        train_r2_dt_tuned
    ],
    'CV RMSE': [
        cv_rmse_lr,
        cv_rmse_ridge,
        -ridge_random_search.best_score_,
        -lasso_random_search.best_score_,
        cv_rmse_dt,
        -dt_grid_search.best_score_
    ]
})

# Calculate improvement
comparison_df['Overfitting'] = comparison_df['CV RMSE'] - comparison_df['Train RMSE']

print("\n📊 COMPREHENSIVE MODEL COMPARISON:")
print("="*80)
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: CV RMSE comparison
models = comparison_df['Model']
cv_rmse_values = comparison_df['CV RMSE']
colors = ['lightblue', 'lightgreen', 'darkgreen', 'orange', 'lightcoral', 'darkred']

axes[0].barh(range(len(models)), cv_rmse_values, color=colors, alpha=0.8)
axes[0].set_yticks(range(len(models)))
axes[0].set_yticklabels(models)
axes[0].set_xlabel('Cross-Validation RMSE (Lower is Better)', fontsize=12)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
for i, v in enumerate(cv_rmse_values):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10)

# Plot 2: Train vs CV RMSE (overfitting check)
x = np.arange(len(models))
width = 0.35

axes[1].bar(x - width/2, comparison_df['Train RMSE'], width, label='Train RMSE', alpha=0.8)
axes[1].bar(x + width/2, comparison_df['CV RMSE'], width, label='CV RMSE', alpha=0.8)
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('Train vs CV RMSE (Overfitting Check)', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# ============================================================================
# STEP 4: IMPROVEMENT ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("STEP 4: IMPROVEMENT ANALYSIS")
print("="*80)

print("\n🎯 IMPROVEMENTS FROM HYPERPARAMETER TUNING:")
print("-" * 80)

# Ridge improvement
ridge_default_cv = cv_rmse_ridge
ridge_tuned_cv = -ridge_random_search.best_score_
ridge_improvement = ((ridge_default_cv - ridge_tuned_cv) / ridge_default_cv) * 100

print(f"\n1. RIDGE REGRESSION:")
print(f"   Default CV RMSE:  {ridge_default_cv:.6f}")
print(f"   Tuned CV RMSE:    {ridge_tuned_cv:.6f}")
print(f"   Improvement:      {ridge_improvement:.2f}%")
print(f"   Best alpha:       {ridge_random_search.best_params_['alpha']:.4f}")

# Lasso results
lasso_tuned_cv = -lasso_random_search.best_score_
print(f"\n2. LASSO REGRESSION:")
print(f"   Tuned CV RMSE:    {lasso_tuned_cv:.6f}")
print(f"   Best alpha:       {lasso_random_search.best_params_['alpha']:.6f}")
print(f"   Features used:    {non_zero_coefs} / {len(best_lasso.coef_)}")

# Decision Tree improvement
dt_default_cv = cv_rmse_dt
dt_tuned_cv = -dt_grid_search.best_score_
dt_improvement = ((dt_default_cv - dt_tuned_cv) / dt_default_cv) * 100

print(f"\n3. DECISION TREE:")
print(f"   Default CV RMSE:  {dt_default_cv:.6f}")
print(f"   Tuned CV RMSE:    {dt_tuned_cv:.6f}")
print(f"   Improvement:      {dt_improvement:.2f}%")
print(f"   Best parameters:")
for param, value in dt_grid_search.best_params_.items():
    print(f"     - {param}: {value}")

# ============================================================================
# STEP 5: SELECT BEST MODEL AND ANALYZE
# ============================================================================
print("\n" + "="*80)
print("STEP 5: SELECT BEST MODEL AND FINAL ANALYSIS")
print("="*80)

best_model_idx = comparison_df['CV RMSE'].idxmin()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_cv_rmse = comparison_df.loc[best_model_idx, 'CV RMSE']

print(f"\n✅ BEST OVERALL MODEL: {best_model_name}")
print(f"   CV RMSE: {best_cv_rmse:.6f}")
print(f"   Train RMSE: {comparison_df.loc[best_model_idx, 'Train RMSE']:.6f}")
print(f"   Train R²: {comparison_df.loc[best_model_idx, 'Train R²']:.6f}")

# Determine which tuned model to use
if 'Ridge (tuned)' in best_model_name:
    final_model = best_ridge
    model_type = 'Ridge'
elif 'Lasso (tuned)' in best_model_name:
    final_model = best_lasso
    model_type = 'Lasso'
elif 'Decision Tree (tuned)' in best_model_name:
    final_model = best_dt
    model_type = 'Decision Tree'
else:
    # Fallback to best tuned model
    if -ridge_random_search.best_score_ <= -lasso_random_search.best_score_ and -ridge_random_search.best_score_ <= -dt_grid_search.best_score_:
        final_model = best_ridge
        model_type = 'Ridge'
    elif -lasso_random_search.best_score_ <= -dt_grid_search.best_score_:
        final_model = best_lasso
        model_type = 'Lasso'
    else:
        final_model = best_dt
        model_type = 'Decision Tree'

print(f"\n🏆 FINAL SELECTED MODEL: {model_type}")

# Analyze feature importance for final model
if model_type in ['Ridge', 'Lasso']:
    print(f"\nTop 10 Most Important Features (by absolute coefficient):")
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': final_model.coef_,
        'Abs_Coefficient': np.abs(final_model.coef_)
    })
    coef_df_sorted = coef_df.sort_values('Abs_Coefficient', ascending=False)
    print(coef_df_sorted.head(10).to_string(index=False))
else:
    print(f"\nTop 10 Most Important Features (by feature importance):")
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': final_model.feature_importances_
    })
    importance_df_sorted = importance_df.sort_values('Importance', ascending=False)
    print(importance_df_sorted.head(10).to_string(index=False))

# ============================================================================
# STEP 6: SUMMARY
# ============================================================================
print("\n" + "="*80)
print("STEP 6: SUMMARY AND CONCLUSIONS")
print("="*80)

print("\n📊 HYPERPARAMETER TUNING SUMMARY:")
print("-" * 80)

print(f"\n✅ Successfully tuned 3 model types:")
print(f"   1. Ridge Regression (RandomizedSearchCV - 100 iterations)")
print(f"   2. Lasso Regression (RandomizedSearchCV - 100 iterations)")
print(f"   3. Decision Tree (GridSearchCV - {7*4*4*3} combinations)")

print(f"\n🎯 KEY FINDINGS:")

print(f"\n1. REGULARIZATION IMPACT:")
print(f"   - Ridge regularization improved CV RMSE by {ridge_improvement:.2f}%")
print(f"   - Optimal alpha for Ridge: {ridge_random_search.best_params_['alpha']:.4f}")
print(f"   - Lasso performed feature selection, keeping {non_zero_coefs}/{len(best_lasso.coef_)} features")

print(f"\n2. DECISION TREE CONSTRAINTS:")
print(f"   - Hyperparameter tuning improved CV RMSE by {dt_improvement:.2f}%")
print(f"   - Optimal max_depth prevents overfitting")
print(f"   - Tuned tree: depth={best_dt.get_depth()}, leaves={best_dt.get_n_leaves()}")
print(f"   - Default tree: depth=24, leaves=1411 (severe overfitting)")

print(f"\n3. BEST MODEL: {model_type}")
print(f"   - CV RMSE: {best_cv_rmse:.6f}")
print(f"   - Shows best balance of bias-variance tradeoff")
print(f"   - Ready for test set predictions")

print("\n4. GENERALIZATION:")
tuned_models_cv = [
    -ridge_random_search.best_score_,
    -lasso_random_search.best_score_,
    -dt_grid_search.best_score_
]
print(f"   - All tuned models have similar CV performance")
print(f"   - CV RMSE range: {min(tuned_models_cv):.6f} - {max(tuned_models_cv):.6f}")
print(f"   - Ensemble methods could potentially improve further")

print("\n" + "="*80)
print("HYPERPARAMETER TUNING COMPLETE - READY FOR SECTION 7")
print("="*80)

print("\n💾 Models ready for Section 7 (Predictions):")
print(f"   - final_model: Best tuned model ({model_type})")
print(f"   - best_ridge: Tuned Ridge model")
print(f"   - best_lasso: Tuned Lasso model")
print(f"   - best_dt: Tuned Decision Tree model")

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

# 9. Submit your predictions to the competition

In [ ]:
import numpy as np
import pandas as pd

print("="*80)
print("SECTION 9: SUBMIT YOUR PREDICTIONS")
print("="*80)

test_predictions_log = final_model.predict(X_test_encoded)
test_predictions = np.exp(test_predictions_log)

submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_predictions
    ).to_csv(
        "submission.csv",
        index=True,
    )
)

print("\n✅ Submission file created: submission.csv")
print(f"Test samples: {len(test_predictions)}")
print(f"Price range: [${test_predictions.min():,.0f}, ${test_predictions.max():,.0f}]")
print(f"Median: ${np.median(test_predictions):,.0f}")

submission_check = pd.read_csv("submission.csv")
print(f"\nFirst 5 predictions:")
print(submission_check.head())

print("\n" + "="*80)

In [ ]:
%%bash

cat submission.csv | head -n 5